# 11. 자료구조 - 트리 표현과 변환

트리 하나를 딕셔너리, 인접 행렬, 1차원 리스트로 각각 담고 서로 바꾼다.

이 자료는 아래 트리 하나만 쓴다.

- A의 자식은 B, C, D
- B의 자식은 E
- D의 자식은 F, G
- C, E, F, G는 단말

---
## 1. 딕셔너리

- 값의 **0번은 부모**, 1번부터가 자식이다.
- 자식이 없는 노드는 **키를 만들지 않는다.**
- 루트는 부모 자리에 자기 자신을 적는다.

In [ ]:
tree = {
    'A': ['A', 'B', 'C', 'D'],   # 루트라서 부모 자리에 자기 자신
    'B': ['A', 'E'],
    'D': ['A', 'F', 'G'],
}                                # C, E, F, G 는 단말이라 키가 없다

print(tree['D'][1:])             # D의 자식
print(tree['D'][0])              # D의 부모
print(tree.get('C', [])[1:])     # C는 키가 없으므로 get 으로 받는다
print('C' not in tree)           # 단말 판정

### 1-1. 딕셔너리에서 바로 안 나오는 것

노드 전체 목록, 단말의 부모, 자손 전체, 루트는 적혀 있지 않아 찾아내야 한다.

In [ ]:
def all_nodes(d):
    kids = {c for v in d.values() for c in v[1:]}   # 값의 1번부터가 자식
    return sorted(set(d) | kids)                    # 키와 합친다


def find_parent(d, node):
    if node in d:
        return d[node][0]        # 키가 있으면 0번이 곧 부모
    for p in d:                  # 단말은 키가 없어 자식 목록을 뒤져야 한다
        if node in d[p][1:]:
            return p


def find_descendants(d, node):
    out = []
    for c in d.get(node, [])[1:]:       # 단말이면 빈 리스트가 되어 그냥 끝난다
        out.append(c)
        out += find_descendants(d, c)   # 그 자식의 자손도 이어 붙인다
    return out


def find_root(d):
    for node in d:
        if d[node][0] == node:   # 부모가 자기 자신이면 루트
            return node


print(all_nodes(tree))
print(find_parent(tree, 'B'), find_parent(tree, 'F'))
print(find_descendants(tree, 'A'))
print(find_root(tree))

---
## 2. 인접 행렬

행이 부모, 열이 자식이다. 연결되어 있으면 1, 아니면 0.
행렬에는 이름이 없으므로 이름과 번호를 잇는 `nodes` 리스트가 따로 필요하다.

In [ ]:
def dict_to_matrix(d):
    nodes = all_nodes(d)                              # 단말까지 포함한 전체 목록
    idx = {name: i for i, name in enumerate(nodes)}   # 이름 -> 번호
    n = len(nodes)
    m = [[0] * n for _ in range(n)]                   # n x n 을 0으로 채운다

    for p in d:
        for c in d[p][1:]:
            m[idx[p]][idx[c]] = 1                     # 행=부모, 열=자식

    return nodes, m


def matrix_to_dict(nodes, m):
    d = {}
    for i in range(len(nodes)):
        kids = [nodes[j] for j in range(len(nodes)) if m[i][j] == 1]
        if not kids:
            continue                                  # 자식이 없으면 키를 만들지 않는다

        parent = nodes[i]                             # 못 찾으면 루트 -> 자기 자신
        for r in range(len(nodes)):
            if m[r][i] == 1:                          # i열에서 1인 행이 부모다
                parent = nodes[r]
                break

        d[nodes[i]] = [parent] + kids
    return d


nodes, m = dict_to_matrix(tree)

print('  ' + ' '.join(nodes))
for i in range(len(nodes)):
    print(nodes[i] + ' ' + ' '.join(str(x) for x in m[i]))

### 2-1. 왕복시켜 보기

In [ ]:
print(matrix_to_dict(nodes, m) == tree)        # 이 트리는 그대로 돌아온다

# 형제 순서가 이름 오름차순이 아니면 왕복에서 순서가 바뀐다
t = {'A': ['A', 'C', 'B']}
n2, m2 = dict_to_matrix(t)

print(matrix_to_dict(n2, m2))
print(matrix_to_dict(n2, m2) == t)             # False

행렬의 한 행은 열 번호 순서로만 읽히므로 형제를 늘어놓은 순서를 담지 못한다.
변환이 항상 안전한 것은 아니다.

---
## 3. 1차원 리스트 - 이진 트리

루트를 1번에 두면 자리를 계산으로 찾을 수 있다. 0번은 쓰지 않는다.

| 대상 | 공식 |
| --- | --- |
| 왼쪽 자식 | `i * 2` |
| 오른쪽 자식 | `i * 2 + 1` |
| 부모 | `i // 2` |

In [ ]:
bin_tree = [None, 'A', 'B', 'C', None, 'D', 'E']   # 0번은 쓰지 않는다


def show(t, i):
    left  = t[i * 2]     if i * 2     < len(t) else None   # 왼쪽 자식
    right = t[i * 2 + 1] if i * 2 + 1 < len(t) else None   # 오른쪽 자식
    up    = t[i // 2]    if i // 2    >= 1     else None   # 부모 (0번은 루트의 부모 자리라 제외)
    print(t[i], '/ 부모', up, '/ 왼쪽', left, '/ 오른쪽', right)


show(bin_tree, 1)
show(bin_tree, 2)
show(bin_tree, 3)

### 3-1. 빈 자리를 표시해야 한다

자식 리스트에서는 **위치가 곧 자리**다. 왼쪽이 비고 오른쪽만 있으면 그 사실을 적어야 한다.
비어 있는 자리는 `None`으로 두고, 뒤쪽이 통째로 비면 생략한다.

In [ ]:
bin_dict = {
    'A': ['A', 'B', 'C'],
    'B': ['A', None, 'D'],   # 왼쪽 자리가 비었다. 자리를 지키려고 None 을 넣는다
    'C': ['A', 'E'],         # 오른쪽 자리가 비었다. 뒤쪽이라 생략했다
}

print(find_root(bin_dict))   # D, E 는 단말이라 키가 없다

---
## 4. 1차원 리스트 - k진 트리

자식을 최대 k개 두는 경우로 넓히면 공식은 다음과 같다. `c`는 몇 번째 자식인지를 0부터 센 값이다.

| 대상 | 공식 |
| --- | --- |
| c번째 자식 | `k * (i - 1) + 2 + c` |
| 부모 | `(i - 2) // k + 1` |

`k = 2`를 넣으면 `i * 2`, `i * 2 + 1`, `i // 2`가 되어 앞 절의 공식과 같아진다.

In [ ]:
def dict_to_array(d, k):
    root = find_root(d)
    arr = [None, root]                       # 0번은 안 쓰고 1번이 루트
    q = [1]                                  # 자리를 정한 노드의 인덱스

    while q:
        i = q.pop(0)
        for c, child in enumerate(d.get(arr[i], [])[1:]):   # 단말이면 꺼낼 자식이 없다
            if child is None:                # 비워 둔 자리는 건너뛴다
                continue
            j = k * (i - 1) + 2 + c          # c번째 자식이 앉을 자리
            while len(arr) <= j:             # 자리가 모자라면 늘린다
                arr.append(None)
            arr[j] = child
            q.append(j)

    return arr


def array_to_dict(arr, k):
    d = {}
    for i in range(1, len(arr)):
        if arr[i] is None:                   # 빈 자리는 노드가 아니다
            continue

        kids = []
        for c in range(k):
            j = k * (i - 1) + 2 + c
            kids.append(arr[j] if j < len(arr) else None)

        while kids and kids[-1] is None:     # 뒤쪽의 빈 자리는 잘라낸다
            kids.pop()

        if not kids:
            continue                         # 자식이 없으면 키를 만들지 않는다

        p = (i - 2) // k + 1                 # 부모 자리
        parent = arr[p] if p >= 1 else arr[i]   # 루트는 자기 자신
        d[arr[i]] = [parent] + kids
    return d

### 4-1. k = 2 로 이진 트리 담기

In [ ]:
arr2 = dict_to_array(bin_dict, 2)

print(arr2)
print(arr2 == bin_tree)                    # 3절에서 손으로 적은 리스트와 같다
print(array_to_dict(arr2, 2) == bin_dict)  # 왕복해도 그대로다

### 4-2. k = 3 으로 맨 위의 트리 담기

k는 최대 자식 수로 정한다. 값에는 부모가 한 칸 끼어 있으므로 1을 뺀다.

In [ ]:
k = max(len(v) - 1 for v in tree.values())   # 필요한 k = 최대 자식 수
print(k)

arr3 = dict_to_array(tree, k)
print(arr3)
print(array_to_dict(arr3, k) == tree)   # 왕복해도 그대로다

### 4-3. 빈 칸

In [ ]:
used = sum(1 for x in arr3 if x is not None)

print(len(arr3) - 1, '칸 중', used, '칸 사용')   # 0번은 세지 않는다

k가 커지거나 트리가 깊어질수록 빈 칸이 빠르게 늘어난다.
인접 행렬이 `n x n`을 잡아 두고 대부분을 0으로 두는 것과 같은 문제다.

---
## 5. 정리

| | 인접 행렬 | 딕셔너리 | 1차원 리스트 |
| --- | --- | --- | --- |
| 자식 수 제한 | 없음 | 없음 | k개 |
| 형제 순서 | 담지 못함 | 담음 | 담음 |
| 노드 목록 | `nodes` 리스트 필요 | 키와 값을 합쳐야 한다 | 값이 곧 이름 |
| 크기 | `n * n` | 자식이 있는 노드 수만큼 | k와 깊이에 따라 급증 |
| 부모 찾기 | 열 전체 탐색 | 키가 있으면 바로, 단말은 탐색 | 계산으로 바로 |

자식이 많은 트리를 1차원 리스트에 담으려면 k를 그만큼 키워야 하고 빈 칸이 폭증한다.
**왼쪽 자식 - 오른쪽 형제**(LCRS)라는 표현은 왼쪽에 첫 자식을, 오른쪽에 다음 형제를 두어
자식이 몇 개든 칸 2개로 처리한다. 대학 자료구조에서 다룬다.